# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madihakomal75/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

### Research Question & Decision Support Goal

* **Primary Question:** Can historical search performance metrics (impressions, average rank position, click-through rate) and temporal freshness indicators (`days_since_last_update`, `content_age_days`) accurately predict organic traffic decay (`trend_direction == 'down'`) on content pages?
* **Decision Supported:** This analysis directly powers an automated editorial prioritization queue, enabling content teams to allocate limited rewriting and refresh bandwidth to high-impact pages facing severe organic decay.

In [1]:
import os, sys, subprocess

REPO_URL = "https://github.com/madihakomal75/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if "google.colab" in sys.modules:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(f"Capstone environment initialized.")
print(f"Dataset Size: {len(df):,} pages | Base Outcome Rate (Decline): {df['is_declining_label'].mean():.2%}")

Capstone environment initialized.
Dataset Size: 30,000 pages | Base Outcome Rate (Decline): 54.21%


### Dataset Scope, Date Windows, & Boundary Exclusion Criteria

* **Data Release:** `content_refresh_anonymized.csv` (30,000 anonymized page records).
* **Observation Window:** 90-day aggregate search metrics derived from Google Search Console (`impressions_90d`, `clicks_90d`, `avg_position`, `ctr`).
* **Excluded Fields & Privacy Boundaries:**
  * **Anonymization:** All raw page URLs and client domains were anonymized into secure hash identifiers (`url_hash_id`, `client_hash_id`).
  * **`search_volume`:** Excluded due to negligible correlation ($r \approx 0.001$) with actual observed page traffic.
  * **Target Leakage Columns:** `trend_direction` was excluded from feature matrices as it defines the classification target.

In [2]:
# Feature Engineering & Matrix Construction
df["staleness_ratio"] = (df["days_since_last_update"] / (df["content_age_days"] + 1)).clip(0, 1)
df["log_impressions"] = np.log1p(df["impressions_90d"].fillna(0))

features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count", "staleness_ratio", "log_impressions"]
X = df[features].fillna(df[features].median())
y = df["is_declining_label"]

print("Feature Vector Constructed:")
print(f"  Shape: {X.shape[0]:,} rows x {X.shape[1]} features")
print(f"  Missing values remaining: {X.isnull().sum().sum()}")

Feature Vector Constructed:
  Shape: 30,000 rows x 8 features
  Missing values remaining: 0


### Experimental Methodology & Honest Validation Design

* **Model Choice:** Gradient Boosted Decision Trees (`HistGradientBoostingClassifier`), selected for strong non-linear feature interaction modeling and robustness to heavy-tailed distributions.
* **Validation Split Strategy:** **Client-Grouped Split** (`GroupShuffleSplit` on `client_hash_id`, 80/20 train/test). Grouping by client domain prevents data leakage and memorization of domain-level baseline authority.
* **Baseline Comparison:** Compared against the Week-4 rule-based heuristic (`HIGH_IMP_STALE`: `impressions_90d >= 500` AND `days_since_last_update >= 180`).

In [3]:
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, precision_score

group_col = "client_hash_id" if "client_hash_id" in df.columns else df.columns[0]
groups = df[group_col]

# Execute Grouped Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(X, y, groups))

X_train, y_train = X.iloc[train_idx], y.iloc[train_idx]
X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]
df_val = df.iloc[val_idx].copy()

# Fit GBDT Model
model = HistGradientBoostingClassifier(random_state=42)
model.fit(X_train, y_train)

df_val["model_prob"] = model.predict_proba(X_val)[:, 1]
df_val["baseline_score"] = (df_val["days_since_last_update"] >= 180).astype(int) * (df_val["impressions_90d"] >= 500).astype(int) * df_val["impressions_90d"]

print("Model training and baseline evaluation completed on validation set.")

Model training and baseline evaluation completed on validation set.


### Honest Results: Model vs. Heuristic Baseline

We evaluate performance using **Precision@50** (the precision among the top 50 prioritized pages) and overall **ROC-AUC** on unseen client domains:

In [4]:
# Compute Precision@50
top50_model = df_val.sort_values(by="model_prob", ascending=False).head(50)
top50_base = df_val.sort_values(by="baseline_score", ascending=False).head(50)

p50_model = top50_model["is_declining_label"].mean()
p50_base = top50_base["is_declining_label"].mean()
auc_model = roc_auc_score(y_val, df_val["model_prob"])

results_df = pd.DataFrame([
    {"Approach": "Heuristic Rule Baseline (HIGH_IMP_STALE)", "Precision@50": f"{p50_base:.1%}", "ROC-AUC": "N/A (Rule)"},
    {"Approach": "Capstone Machine Learning Model (GBDT)", "Precision@50": f"{p50_model:.1%}", "ROC-AUC": f"{auc_model:.4f}"}
])

print("Honest Validation Results Table:")
results_df

Honest Validation Results Table:


,Approach,Precision@50,ROC-AUC
0,Heuristic Rule Baseline (HIGH_IMP_STALE),66.0%,N/A (Rule)
1,Capstone Machine Learning Model (GBDT),94.0%,0.7808


### Boundaries & Model Limitations

1. **Non-Causal Signals:** Predictions reflect historical correlations with traffic decay; refreshing content on flagged pages does not guarantee immediate ranking recovery.
2. **Seasonal Distortion Risk:** Pages experiencing expected seasonal off-peak drops may be incorrectly flagged as stale/declining.
3. **No-Go Boundary (Top-3 Protection):** Healthy pages ranking in positions 1–3 must be manually excluded from refresh queues to avoid destabilizing strong organic search placements.

In [5]:
# Measure Top-3 Protection Candidates in Validation Queue
top_ranked_flagged = df_val[(df_val["model_prob"] >= 0.70) & (df_val["avg_position"] <= 3.0)]

print(f"Validation Pages Flagged by Model needing Top-3 Protection: {len(top_ranked_flagged)} pages")
print("Guideline: These pages must be routed to human review rather than automated editing queues.")

Validation Pages Flagged by Model needing Top-3 Protection: 91 pages
Guideline: These pages must be routed to human review rather than automated editing queues.


### Content Action Playbook & Prioritized Queue

Each candidate page is mapped to a explicit recommendation category and reason code:

* **`REFRESH_URGENT`** ($P \ge 0.70$): Stale, high-impression page with active rank decay.
* **`MONITOR_RANK`** ($0.45 \le P < 0.70$): Minor ranking decay; monitor weekly.
* **`NO_ACTION_HEALTHY`** ($P < 0.45$): Stable performance; preserve current structure.

In [6]:
def assign_action(prob):
    if prob >= 0.70:
        return "REFRESH_URGENT"
    elif prob >= 0.45:
        return "MONITOR_RANK"
    else:
        return "NO_ACTION_HEALTHY"

df["capstone_model_prob"] = model.predict_proba(X)[:, 1]
df["recommended_action"] = df["capstone_model_prob"].apply(assign_action)

print("Final Action Recommendation Distribution across Full Dataset:")
df["recommended_action"].value_counts()

Final Action Recommendation Distribution across Full Dataset:


,count
recommended_action,
MONITOR_RANK,11948
NO_ACTION_HEALTHY,9503
REFRESH_URGENT,8549


### Generated Research Artifacts & Deliverables

We generate and save artifacts to `work/outputs/` for embedding directly in the deployed research paper.

In [7]:
os.makedirs("work/outputs", exist_ok=True)

id_col = "url_hash_id" if "url_hash_id" in df.columns else ("page_id" if "page_id" in df.columns else df.columns[0])

# Export Final Ranked Queue CSV
df_capstone_ranked = df.sort_values(by="capstone_model_prob", ascending=False).reset_index(drop=True)
df_capstone_ranked["rank"] = df_capstone_ranked.index + 1

output_cols = [id_col, "rank", "capstone_model_prob", "recommended_action", "avg_position", "days_since_last_update", "impressions_90d"]
df_capstone_ranked[output_cols].to_csv("work/outputs/capstone_ranked_queue.csv", index=False)

print(f"Successfully exported final capstone queue ({len(df_capstone_ranked):,} rows) to work/outputs/capstone_ranked_queue.csv")

# Print ML-12 Closing Summaries
print("\n" + "="*60)
print("ML-12 CAPSTONE DEMO & SUMMARY ARTIFACTS")
print("="*60)

print("\n1. 5-MINUTE DEMO OUTLINE:")
print("   - 0:00-1:00: Problem & Business Impact (Why editorial teams waste budget on wrong pages).")
print("   - 1:00-2:00: Data & Honest Split Design (Grouped by client domain to prevent leakage).")
print("   - 2:00-3:30: Model Results vs Baseline (Precision@50 and GBDT lift over heuristic rules).")
print("   - 3:30-5:00: Action Playbook & Safety Rules (No-Go list & operationalizing REFRESH_URGENT).")

print("\n2. SOCIAL POST COPY:")
print("   🚀 Excited to share my capstone project on Search Intelligence ML with FlyRank! Built a decision-support model that identifies organic search decay with 80%+ Precision@50 using client-grouped validation splits. Check out my GitHub repo for the full open-source playbook: https://github.com/madihakomal75/flyrank-ml-internship")

print("\n3. EMPLOYER-FACING SUMMARY (3 SENTENCES):")
print("   Developed an end-to-end search performance decay classification model that prioritizes content refresh queues for enterprise websites. Implemented client-grouped validation splits to ensure zero target leakage and robust domain generalizability. Delivered an actionable editorial decision-support playbook that improves candidate selection precision over traditional heuristic rules.")

Successfully exported final capstone queue (30,000 rows) to work/outputs/capstone_ranked_queue.csv

ML-12 CAPSTONE DEMO & SUMMARY ARTIFACTS

1. 5-MINUTE DEMO OUTLINE:
   - 0:00-1:00: Problem & Business Impact (Why editorial teams waste budget on wrong pages).
   - 1:00-2:00: Data & Honest Split Design (Grouped by client domain to prevent leakage).
   - 2:00-3:30: Model Results vs Baseline (Precision@50 and GBDT lift over heuristic rules).
   - 3:30-5:00: Action Playbook & Safety Rules (No-Go list & operationalizing REFRESH_URGENT).

2. SOCIAL POST COPY:
   🚀 Excited to share my capstone project on Search Intelligence ML with FlyRank! Built a decision-support model that identifies organic search decay with 80%+ Precision@50 using client-grouped validation splits. Check out my GitHub repo for the full open-source playbook: https://github.com/madihakomal75/flyrank-ml-internship

3. EMPLOYER-FACING SUMMARY (3 SENTENCES):
   Developed an end-to-end search performance decay classification mo